# Mini-Challenge 2 — Margin per case, the vectorised way
### Saint Mary's · Thursday morning, 11:00

> *"For all 320 encounters, compute the margin (DRG reimbursement minus actual cost) per case. **No Python for-loop**. Total margin and number of loss-making cases on my desk."*

**Time:** 25 min · **Mode:** tandem · **Goal:** practise vectorised arithmetic on real data.

---

# Solution — Mini-Challenge 2

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# Walk up to find the 'data/' folder — works from any notebook location
HERE = Path.cwd()
DATA = next((p / "data" for p in [HERE, *HERE.parents] if (p / "data").exists()), None)
assert DATA is not None, "data/ folder not found"
print("Using DATA =", DATA)

In [ ]:
enc = pd.read_csv(DATA / "encounters.csv", parse_dates=["admission_date", "discharge_date"])
drg = pd.read_csv(DATA / "drg_catalog.csv")

enc = enc.merge(drg, on="drg_code", how="left")
enc["margin_eur"] = enc["base_reimbursement_eur"] - enc["actual_cost_eur"]

print(f"Total margin: € {enc['margin_eur'].sum():,.2f}")
print(f"Loss-making cases: {(enc['margin_eur'] < 0).sum()} of {len(enc)}")

## Fast-Track — Top 3 loss-making DRGs

In [ ]:
losing = enc[enc["margin_eur"] < 0]
top_loss = (losing.groupby(["drg_code", "label"])
                  .agg(count=("encounter_id", "count"),
                       total_margin_eur=("margin_eur", "sum"))
                  .sort_values("total_margin_eur")
                  .head(3))
top_loss